In [4]:
import os

if not os.path.exists("f1_cache"):
    os.makedirs("f1_cache")
    print("✅ f1_cache directory created")
else:
    print("✅ f1_cache already exists")


✅ f1_cache directory created


In [5]:
import fastf1
fastf1.Cache.enable_cache("f1_cache")


In [13]:
import fastf1
import pandas as pd
import joblib

# Enable cache
fastf1.Cache.enable_cache("f1_cache")

# Load trained model
model = joblib.load("model.joblib")

# Same features as training
features = ['avg_race_pace','max_stint','pitstops','quali_pace','grid_position']

def predict_race(year, gp):
    print(f"🏁 Predicting {year} {gp} winner...\n")

    race = fastf1.get_session(year, gp, "R")
    quali = fastf1.get_session(year, gp, "Q")

    race.load()
    quali.load()

    laps = race.laps
    results = race.results

    avg_race_pace = laps.groupby("Driver")["LapTime"].mean()
    max_stint = laps.groupby("Driver")["Stint"].max()
    pitstops = laps.groupby("Driver")["PitInTime"].count()
    quali_pace = quali.laps.groupby("Driver")["LapTime"].min()
    grid_pos = results.set_index("Abbreviation")["GridPosition"]

    df = pd.DataFrame({
        "avg_race_pace": avg_race_pace,
        "max_stint": max_stint,
        "pitstops": pitstops,
        "quali_pace": quali_pace,
        "grid_position": grid_pos
    }).reset_index().rename(columns={"index":"Driver"})

    # Convert times to seconds
    for col in ['avg_race_pace','quali_pace']:
        df[col] = pd.to_timedelta(df[col], errors='coerce').dt.total_seconds()

    df = df.dropna()

    X_pred = df[features]

    df['win_probability'] = model.predict_proba(X_pred)[:,1]
    df = df.sort_values("win_probability", ascending=False)

    print(f"🏆 TOP 3 {year} {gp} WINNER PREDICTION:\n")
    print(df[['Driver','win_probability']].head(3))

    return df


In [14]:
predict_race(2025, "azerbaijan")

🏁 Predicting 2025 azerbaijan winner...



core           INFO 	Loading data for Azerbaijan Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 1 completed the race distance 00:00.015000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '63', '55', '12', '30', '22', '4', '44', '16', '6', '5', '8

🏆 TOP 3 2025 azerbaijan WINNER PREDICTION:

   Driver  win_probability
19    VER         0.337820
10    LAW         0.136200
16    SAI         0.116829


,Driver,avg_race_pace,max_stint,pitstops,quali_pace,grid_position,win_probability
19,VER,106.058771,2.0,1,101.117,1.0,0.337820
10,LAW,106.787542,2.0,1,101.537,3.0,0.136200
16,SAI,106.461042,2.0,1,101.595,2.0,0.116829
18,TSU,106.922458,2.0,1,101.788,6.0,0.066366
2,ANT,106.568229,2.0,1,101.464,4.0,0.053886
7,HAD,107.080292,2.0,1,101.647,8.0,0.021725
15,RUS,106.456563,2.0,1,101.455,5.0,0.019721
12,NOR,106.974146,2.0,1,101.322,7.0,0.009786
11,LEC,107.075688,2.0,1,101.458,10.0,0.009741
13,OCO,108.077833,3.0,2,103.004,20.0,0.006957


In [15]:
predict_race(2025,"Monaco")

logger      WARNING 	Failed to load schedule from FastF1 backend!
req            INFO 	No cached data found for season_schedule. Loading data...
_api           INFO 	Fetching season schedule...


🏁 Predicting 2025 Monaco winner...



req            INFO 	Data has been written to cache!
core           INFO 	Loading data for Monaco Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '16', '81', '1', '44', '6', '31', '30', '23', '55', '63', '87', '43', '5', '18', '27', '22', '12', '14', '10']
core      

🏆 TOP 3 2025 Monaco WINNER PREDICTION:

   Driver  win_probability
12    NOR         0.495923
11    LEC         0.144111
14    PIA         0.047116


,Driver,avg_race_pace,max_stint,pitstops,quali_pace,grid_position,win_probability
12,NOR,77.356962,3.0,2,69.954,1.0,0.495923
11,LEC,77.397103,3.0,2,70.063,2.0,0.144111
14,PIA,77.403859,3.0,2,70.129,3.0,0.047116
0,ALB,79.548842,3.0,2,70.732,10.0,0.041966
19,VER,77.620705,3.0,2,70.669,4.0,0.016192
10,LAW,79.228429,3.0,2,71.129,9.0,0.012844
13,OCO,79.219117,3.0,2,70.942,8.0,0.011593
7,HAD,79.206818,3.0,2,70.923,5.0,0.011569
15,RUS,79.837987,3.0,3,71.507,14.0,0.009228
8,HAM,78.015769,3.0,2,70.382,7.0,0.008115


In [17]:
predict_race(2025,"Las Vegas")

🏁 Predicting 2025 Las Vegas winner...



core           INFO 	Loading data for Las Vegas Grand Prix - Race [v3.7.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No 

🏆 TOP 3 2025 Las Vegas WINNER PREDICTION:

   Driver  win_probability
19    VER         0.294092
12    NOR         0.272703
16    SAI         0.257912


,Driver,avg_race_pace,max_stint,pitstops,quali_pace,grid_position,win_probability
19,VER,97.368580,2.0,1,108.257,2.0,0.294092
12,NOR,97.783400,2.0,1,107.934,1.0,0.272703
16,SAI,98.067060,2.0,1,108.296,3.0,0.257912
15,RUS,97.839500,2.0,1,108.803,4.0,0.209250
14,PIA,97.921580,2.0,1,108.961,5.0,0.086040
10,LAW,99.422857,3.0,2,109.062,6.0,0.081998
6,GAS,99.202620,2.0,1,111.540,10.0,0.038950
11,LEC,97.982140,2.0,1,109.872,9.0,0.026257
7,HAD,98.273720,2.0,1,109.554,8.0,0.026257
1,ALO,99.074740,2.0,1,109.466,7.0,0.026257
